# RSF-PHATE quickstart

This notebook shows a minimal end-to-end use of the public `rsfphate` package on a synthetic survival dataset.

In [ ]:
from rsfphate import RSFPhate, make_donut_survival

In [ ]:
X, time, event, truth = make_donut_survival(
    n_samples=1000,
    censoring_fraction=0.10,
    random_state=42,
)

model = RSFPhate(
    n_clusters=2,
    n_estimators=100,
    diffusion_time=6.0,
    random_state=42,
)

labels = model.fit_predict(X, time, event)

In [ ]:
print("Embedding shape:", model.embedding_.shape)
print("Proximity shape:", model.proximity_.shape)
print("Smoothed proximity shape:", model.smoothed_proximity_.shape)
print("First 10 predicted labels:", labels[:10])
print("First 10 ground-truth labels:", truth[:10])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=labels, s=25)
plt.title("Covariates")
plt.xticks([])
plt.yticks([])
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(model.embedding_[:, 0], model.embedding_[:, 1], c=labels, s=25)
plt.title("RSF-PHATE embedding")
plt.xticks([])
plt.yticks([])
plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.nonparametric import kaplan_meier_estimator
from rsfphate import to_survival_array
import numpy as np


def empirical_pbox(times, events):
    """Return lower and upper empirical p-box envelopes on the observed time grid."""
    grid = np.unique(times)
    lower = np.array([np.mean((times <= t) & events) for t in grid], dtype=float)
    upper = np.array([np.mean(times <= t) for t in grid], dtype=float)
    return grid, lower, upper


def plot_cluster_survival_summary(labels_to_plot, title):
    unique_labels = np.unique(labels_to_plot)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for cluster in unique_labels:
        mask = labels_to_plot == cluster
        km_time, km_surv = kaplan_meier_estimator(event[mask], time[mask])
        axes[0].step(
            np.r_[0.0, km_time],
            np.r_[1.0, km_surv],
            where="post",
            label=f"Cluster {cluster} (n={mask.sum()})",
        )

        pbox_time, lower, upper = empirical_pbox(time[mask], event[mask])
        axes[1].step(pbox_time, lower, where="post", linewidth=2)
        axes[1].step(pbox_time, upper, where="post", linestyle="--", linewidth=2)
        axes[1].fill_between(pbox_time, lower, upper, step="post", alpha=0.18)

        event_rate = float(np.mean(event[mask]))
        print(f"Cluster {cluster}: n={mask.sum()}, observed-event rate={event_rate:.3f}")

    axes[0].set_title(f"{title}: Kaplan-Meier")
    axes[0].set_xlabel("Observed time")
    axes[0].set_ylabel("Survival probability")
    axes[0].legend()

    axes[1].set_title(f"{title}: empirical p-box")
    axes[1].set_xlabel("Observed time")
    axes[1].set_ylabel("Cumulative incidence bounds")
    axes[1].set_ylim(0.0, 1.0)

    plt.tight_layout()
    plt.show()


print("ARI against ground truth:", round(adjusted_rand_score(truth, labels), 3))
plot_cluster_survival_summary(labels, "RSF-PHATE clusters")

In [ ]:
# Baseline: Cox risk-score clustering

y_struct = to_survival_array(time, event)
cox = CoxPHSurvivalAnalysis(alpha=0.0)
cox.fit(X, y_struct)
risk_scores = cox.predict(X).reshape(-1, 1)
baseline_labels = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(risk_scores)

In [ ]:

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=baseline_labels, s=25)
plt.title("Cox risk-score clustering on the original covariates")
plt.xticks([])
plt.yticks([])
plt.show()

In [ ]:
print("Baseline ARI against ground truth:", round(adjusted_rand_score(truth, baseline_labels), 3))
plot_cluster_survival_summary(baseline_labels, "Cox risk-score clustering")